# Total Site and SUGCC Interpretation
This notebook uses the packaged `pulp_mill.json` case to move from local direct targets to the Total Process and Total Site views that drive site-wide utility decisions.

**Support notice:** This advanced notebook exercises unsupported internal owner modules. Only `from OpenPinch.main import pinch_analysis_service` is compatibility protected.


## Case context
`pulp_mill.json` is a shipped multi-area site case. It is large enough to show how zone-level process bottlenecks aggregate into a site utility answer while using the internal `PinchWorkspace` and `PinchProblem` coordinators.


In [ ]:
import pandas as pd

from IPython.display import display

from OpenPinch.application.workspace import PinchWorkspace
from OpenPinch.domain._value.resolution import get_scalar_value


In [ ]:
workspace = PinchWorkspace(source="pulp_mill.json", project_name="Site")
baseline = workspace.use_case("baseline")
baseline.target()
summary = baseline.summary_frame()
detailed_summary = baseline.summary_frame(detailed=True)
catalog = baseline.plot.catalog()

{
    "cases": workspace.list_cases(),
    "graph_count": len(catalog),
    "site_targets": summary.loc[summary["Target"].str.startswith("Site/"), "Target"].tolist(),
}


## Scope ladder: Direct Integration vs Total Process vs Total Site
Read the three site rows together. The direct row answers the purely process-side question, the Total Process row sums zonal utility use, and the Total Site row shows what changes once the utility system is reconciled across the whole mill.


In [ ]:
scope_rows = summary.loc[
    summary["Target"].isin(
        [
            "Site/Direct Integration",
            "Site/Total Process Target",
            "Site/Total Site Target",
        ]
    ),
    [
        "Target",
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
].copy()

scope_rows["Scope"] = ["Direct Integration", "Total Process Target", "Total Site Target"]
scope_rows[[
    "Scope",
    "Hot Utility Target",
    "Cold Utility Target",
    "Heat Recovery",
    "Hot Pinch",
    "Cold Pinch",
]]


In [ ]:
scope_index = detailed_summary.set_index("Target")
pd.DataFrame(
    [
        {
            "Comparison": "Total Process - Direct",
            "Hot Utility Delta": scope_index.loc["Site/Total Process Target", "Qh (value)"] - scope_index.loc["Site/Direct Integration", "Qh (value)"],
            "Cold Utility Delta": scope_index.loc["Site/Total Process Target", "Qc (value)"] - scope_index.loc["Site/Direct Integration", "Qc (value)"],
            "Heat Recovery Delta": scope_index.loc["Site/Total Process Target", "Qr (value)"] - scope_index.loc["Site/Direct Integration", "Qr (value)"],
        },
        {
            "Comparison": "Total Site - Direct",
            "Hot Utility Delta": scope_index.loc["Site/Total Site Target", "Qh (value)"] - scope_index.loc["Site/Direct Integration", "Qh (value)"],
            "Cold Utility Delta": scope_index.loc["Site/Total Site Target", "Qc (value)"] - scope_index.loc["Site/Direct Integration", "Qc (value)"],
            "Heat Recovery Delta": scope_index.loc["Site/Total Site Target", "Qr (value)"] - scope_index.loc["Site/Direct Integration", "Qr (value)"],
        },
    ]
)


## Which local zones drive the site answer?
Before reading site graphs, rank the direct-integration zone rows. This identifies the areas that dominate hot utility demand, cold utility demand, and recovered heat before the utility network changes the answer.


In [ ]:
zone_rows = detailed_summary.loc[
    detailed_summary["Target"].str.endswith("/Direct Integration") & (detailed_summary["Target"] != "Site/Direct Integration"),
    ["Target", "Qh (value)", "Qc (value)", "Qr (value)"],
].rename(
    columns={
        "Qh (value)": "Hot Utility Target",
        "Qc (value)": "Cold Utility Target",
        "Qr (value)": "Heat Recovery",
    }
).copy()
zone_rows["Zone"] = zone_rows["Target"].str.replace("/Direct Integration", "", regex=False)

leader_tables = []
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    leaders = zone_rows.nlargest(5, metric)[["Zone", metric]].reset_index(drop=True)
    leaders.index = leaders.index + 1
    leaders = leaders.rename_axis("Rank").reset_index()
    leaders["Metric"] = metric
    leader_tables.append(leaders[["Metric", "Rank", "Zone", metric]])

pd.concat(leader_tables, ignore_index=True)


## Read one real local GCC, then the site-level graphs
Use one consistent zone screen for the local process view. `Bleaching` is a good local example because it carries a large hot utility burden in the direct summary, while the site-level profiles and SUGCC show what the wider utility system does with that burden.


In [ ]:
display(baseline.plot.grand_composite_curve(zone_name="Bleaching"))


In [ ]:
display(baseline.plot.total_site_profiles(zone_name="Site"))
display(baseline.plot.site_utility_grand_composite_curve(zone_name="Site"))


## Inspect the raw graph data
Rendered figures are only one view. The serialized graph data returned by `plot.get_graph_data()` is the current internal hand-off shape for app and service layers that need structured graph data.


In [ ]:
graph_data = baseline.plot.get_graph_data()

pd.DataFrame(
    [
        {
            "Target": target_name,
            "Target Type": graph_set["target_type"],
            "Graph Types": ", ".join(graph["type"] for graph in graph_set["graphs"]),
            "Zone Address": graph_set["zone_address"],
        }
        for target_name, graph_set in graph_data.items()
        if target_name in ["Bleaching/Direct Integration", "Site/Total Site Target"]
    ]
)


In [ ]:
selected_graph_data = {
    "Bleaching GCC": baseline.plot.grand_composite_curve(
        zone_name="Bleaching",
        return_graph_data=True,
    ),
    "Total Site Profiles": baseline.plot.total_site_profiles(
        zone_name="Site",
        return_graph_data=True,
    ),
    "Site Utility Grand Composite Curve": baseline.plot.site_utility_grand_composite_curve(
        zone_name="Site",
        return_graph_data=True,
    ),
}

pd.DataFrame(
    [
        {
            "Graph": label,
            "Graph Type": graph["type"],
            "Segment Count": len(graph.get("segments", [])),
            "First Segment Keys": ", ".join(sorted(graph["segments"][0].keys())) if graph.get("segments") else None,
        }
        for label, graph in selected_graph_data.items()
    ]
)


## Local and site cogeneration screens
Cogeneration questions depend on which base target family you want to screen. The local `Bleaching` screen is anchored to `Direct Integration`, while the site-level screen is explicitly anchored to `Total Site Target`.


In [ ]:
bleach_problem = workspace.copy_case("baseline", "bleaching_cogeneration_screen", activate=False)
bleach_problem.target()
cogen_bleach = bleach_problem.target.cogeneration(
    zone_name="Bleaching",
    options={"base_target_type": "Direct Integration"},
)

site_problem = workspace.copy_case("baseline", "site_cogeneration_screen", activate=False)
site_problem.target()
cogeneration_target = site_problem.target.cogeneration(
    options={
        "base_target_type": "Total Site Target",
    }
)

pd.DataFrame(
    [
        {
            "Screen": "Bleaching local screen",
            "Target": cogen_bleach.name,
            "Hot Utility Target": (cogen_bleach.hot_utility_target),
            "Cold Utility Target": (cogen_bleach.cold_utility_target),
            "Heat Recovery": (cogen_bleach.heat_recovery_target),
            "Power Cogeneration Target": (cogen_bleach.work_target),
            "Turbine Efficiency Target": (cogen_bleach.turbine_efficiency_target),
        },
        {
            "Screen": "Total-site screen",
            "Target": cogeneration_target.name,
            "Hot Utility Target": (cogeneration_target.hot_utility_target),
            "Cold Utility Target": (cogeneration_target.cold_utility_target),
            "Heat Recovery": (cogeneration_target.heat_recovery_target),
            "Power Cogeneration Target": get_scalar_value(cogeneration_target.work_target),
            "Turbine Efficiency Target": (cogeneration_target.turbine_efficiency_target),
        },
    ]
)


## Next step
This notebook stays on one real packaged site case. Notebook 4 uses the multiperiod derivatives to show how the same site-style targeting surface behaves across named operating periods such as `summer`, `shoulder`, and `winter`.
